# Section 4: Bayesian Hyperparameter Optimisation

Selected model: Random Forest, based on the repository's highest reported Macro F1 and observation Top-1 score. This notebook preserves the training pipeline's seven base features, four circular bearing features, median imputation, and balanced class weights. It never reads evaluator-testing data. Student-testing files may be prepared by the setup using the fixed existing pipeline, but no student features, labels, or metrics are passed to the optimization objective; student metrics are calculated only after the search.

## Search space and method

Optuna TPE, seeded with 42, maximises mean Macro F1 across three fixed GroupKFold splits by trip on a six-trip representative pilot subset. The 30-trial full-data target was reduced to three pilot trials after timing showed that full-data CV exceeded the 30-minute limit. Pilot search distributions: `n_estimators` integer-uniform 50–150, step 50; `max_depth` categorical {None, 4, 8, 12, 16}; `min_samples_split` integer-uniform 2–10; `min_samples_leaf` integer-uniform 1–5; `max_features` categorical {sqrt, log2, 0.5}; `criterion` categorical {gini, entropy}. Class weights remain balanced. 30 full-data trials were planned; the deadline-limited pilot completed three trials after the full-data baseline exceeded the 30-minute budget. These ranges explore model capacity and regularisation around the documented baseline (300 trees, unlimited depth, split 4, leaf 2).

| Hyperparameter | Meaning | Pilot search range |
|---|---|---|
| `n_estimators` | Number of trees; higher values can improve stability but increase fit time. | 50–150, step 50 (official baseline remains 300) |
| `max_depth` | Maximum tree depth; limits model complexity. | None, 4, 8, 12, 16 |
| `min_samples_split` | Minimum samples needed to split a node. | Integer 2–10 |
| `min_samples_leaf` | Minimum samples retained in a leaf; larger values regularize. | Integer 1–5 |
| `max_features` | Features considered at each split; controls tree diversity. | sqrt, log2, 0.5 |
| `criterion` | Split impurity measure. | gini, entropy |

The full-data 300-tree grouped baseline did not complete within the 30-minute run window; it produced no complete CV score. The metrics generated by this notebook are explicitly a representative-subset pilot. The original untuned configuration remains the official baseline.

In [1]:
from pathlib import Path
import json, time, warnings, sys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, f1_score, log_loss
try:
    import optuna
    from optuna.samplers import TPESampler
except ImportError as e:
    raise ImportError('Install Optuna with pip install optuna before running this notebook.') from e
warnings.filterwarnings('ignore')
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
DATA=ROOT/'data'/'processed'; STUDENT=DATA/'student_test'; OUT=ROOT/'outputs'/'hyperparameter_optimization'; OUT.mkdir(parents=True,exist_ok=True)
RAW=Path(r'C:\Users\user\Desktop\5th\ML_LAB\PathTrace\data')  # only train and student_test are used
SEED=42; N_SPLITS=3; N_TRIALS=3  # time-bounded pilot; 30 full-data trials were planned
RESUME_PILOT_CHECKPOINTS=True
BASE=['distance_m','gps_speed_mps','gps_displacement_m','time_gap_s','segment_length_m','heading_difference','projection_fraction']
def features(d):
    X=d[BASE].copy(); g=np.radians(d.gps_bearing); s=np.radians(d.segment_bearing)
    X['gps_bearing_sin']=np.sin(g); X['gps_bearing_cos']=np.cos(g); X['segment_bearing_sin']=np.sin(s); X['segment_bearing_cos']=np.cos(s); return X
def load_labelled(folder, name, trip_ids=None):
    paths=sorted(folder.glob('trip_*_features.csv')); frames=[]; skipped=[]
    if not paths: raise FileNotFoundError(f'Missing {name} feature files in {folder}')
    for p in paths:
        tid=p.stem.replace('trip_','').replace('_features',''); ap=folder/f'trip_{tid}_monotonic_alignment.csv'
        if trip_ids is not None and tid not in trip_ids: continue
        if not ap.exists(): skipped.append(tid); continue
        d=pd.read_csv(p); a=pd.read_csv(ap)
        d=d.merge(a[['trip_id','gps_index','aligned_arc_id']],on=['trip_id','gps_index'],how='inner',validate='many_to_one')
        d['label']=(d.arc_id==d.aligned_arc_id).astype(int); pos=d.groupby(['trip_id','gps_index']).label.transform('max'); d=d[pos.eq(1)]
        if len(d): frames.append(d)
    if skipped: print(f'{name}: skipped {len(skipped)} trips without alignments')
    if not frames: raise ValueError(f'No labelled {name} trips')
    return pd.concat(frames,ignore_index=True)
# Reuse existing project alignment, candidate-generation and feature-extraction code without changing it.
if not (RAW/'train').is_dir() or not (RAW/'student_test').is_dir(): raise FileNotFoundError(f'Expected train and student_test under {RAW}')
sys.path.insert(0,str(ROOT/'src'))
import batch_alignment as alignment_pipeline, batch_feature_extraction as feature_pipeline, candidate_generation
alignment_pipeline.TRAIN_DIR=RAW/'train'; alignment_pipeline.PROCESSED_DIR=DATA
for fp in sorted(DATA.glob('trip_*_features.csv')):
    tid=fp.stem.replace('trip_','').replace('_features',''); ap=DATA/f'trip_{tid}_monotonic_alignment.csv'
    if not ap.exists(): alignment_pipeline.process_trip(tid)
STUDENT.mkdir(parents=True,exist_ok=True)
candidate_generation.TRAIN_DIR=RAW/'student_test'; feature_pipeline.TRAIN_DIR=RAW/'student_test'; feature_pipeline.PROCESSED_DIR=STUDENT
expected_student=[p.name for p in (RAW/'student_test').iterdir() if p.is_dir()]
if not all((STUDENT/f'trip_{tid}_features.csv').exists() for tid in expected_student): feature_pipeline.main()
else: print('Reusing existing student-testing feature tables')
alignment_pipeline.TRAIN_DIR=RAW/'student_test'; alignment_pipeline.PROCESSED_DIR=STUDENT
for fp in sorted(STUDENT.glob('trip_*_features.csv')):
    tid=fp.stem.replace('trip_','').replace('_features',''); ap=STUDENT/f'trip_{tid}_monotonic_alignment.csv'
    if not ap.exists(): alignment_pipeline.process_trip(tid)
train_paths=sorted(DATA.glob('trip_*_features.csv')); by_size=sorted(train_paths,key=lambda p:p.stat().st_size); pick=[round(q*(len(by_size)-1)) for q in (0.05,0.20,0.35,0.50,0.65,0.80)]; PILOT_IDS=[by_size[i].stem.replace('trip_','').replace('_features','') for i in pick]; PILOT_IDS=sorted(set(PILOT_IDS))
train=load_labelled(DATA,'training',PILOT_IDS); X=features(train); y=train.label.astype(int); groups=train.trip_id
print('PILOT SCOPE — representative training trips:',PILOT_IDS)
if groups.nunique()<N_SPLITS: raise ValueError(f'Require at least {N_SPLITS} aligned training trips for GroupKFold; found {groups.nunique()}.')
folds=list(GroupKFold(N_SPLITS).split(X,y,groups)); fold_ids=[{'train':[str(v) for v in sorted(groups.iloc[a].unique())],'validation':[str(v) for v in sorted(groups.iloc[b].unique())]} for a,b in folds]
assert all(not(set(f['train'])&set(f['validation'])) for f in fold_ids)
current_folds=[{'fold':i+1,'train_trip_ids':[str(v).zfill(8) for v in f['train']],'validation_trip_ids':[str(v).zfill(8) for v in f['validation']]} for i,f in enumerate(fold_ids)]
resume_validated=False
checkpoint_files=['baseline_results.csv','optimization_trials.csv','best_parameters.json','best_cv_score.json','optimization_summary.json']
if RESUME_PILOT_CHECKPOINTS and any((OUT/name).exists() for name in checkpoint_files):
    if not all((OUT/name).exists() for name in checkpoint_files): raise RuntimeError('Pilot checkpoint set is incomplete; refusing to mix stale outputs.')
    saved=json.loads((OUT/'optimization_summary.json').read_text())
    if saved.get('pilot_training_trip_ids')!=PILOT_IDS or saved.get('fold_assignments')!=current_folds or saved.get('run_trial_limit')!=N_TRIALS or saved.get('raw_data_root')!=str(RAW):
        raise RuntimeError('Saved pilot checkpoints do not match the current data path, pilot trips, or CV folds; set RESUME_PILOT_CHECKPOINTS=False for a deliberate new run.')
    resume_validated=True
def model(params):
    options={'class_weight':'balanced','n_jobs':-1,'random_state':SEED}; options.update(params)
    return Pipeline([('imputer',SimpleImputer(strategy='median')),('model',RandomForestClassifier(**options))])
def scores(params):
    out=[]
    for a,b in folds:
        m=model(params).fit(X.iloc[a],y.iloc[a]); out.append(f1_score(y.iloc[b],m.predict(X.iloc[b]),average='macro',zero_division=0))
    return out
baseline={'n_estimators':300,'max_depth':None,'min_samples_split':4,'min_samples_leaf':2}
if resume_validated and (OUT/'baseline_results.csv').exists():
    baseline_row=pd.read_csv(OUT/'baseline_results.csv').iloc[0]; base_cv=float(baseline_row.cv_macro_f1_mean); base_scores=json.loads(baseline_row.fold_scores)
else:
    base_scores=scores(baseline); base_cv=float(np.mean(base_scores))
    pd.DataFrame([{'experiment_scope':'pilot subset; official untuned 300-tree model','model':'Untuned Random Forest','cv_macro_f1_mean':base_cv,'cv_macro_f1_std':float(np.std(base_scores,ddof=1)),'fold_scores':json.dumps(base_scores)}]).to_csv(OUT/'baseline_results.csv',index=False)
print('training trips',groups.nunique(),'baseline CV Macro F1',base_cv,'folds',fold_ids)

C:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Reusing existing student-testing feature tables


PILOT SCOPE — representative training trips: ['00000003', '00000022', '00000048', '00000081', '00000083', '00000099']
training trips 6 baseline CV Macro F1 0.8295458914078123 folds [{'train': ['3', '22', '48', '83', '99'], 'validation': ['81']}, {'train': ['22', '48', '81', '83'], 'validation': ['3', '99']}, {'train': ['3', '81', '99'], 'validation': ['22', '48', '83']}]


In [2]:
def objective(t):
    p={'n_estimators':t.suggest_int('n_estimators',50,150,step=50),'max_depth':t.suggest_categorical('max_depth',[None,4,8,12,16]),'min_samples_split':t.suggest_int('min_samples_split',2,10),'min_samples_leaf':t.suggest_int('min_samples_leaf',1,5),'max_features':t.suggest_categorical('max_features',['sqrt','log2',0.5]),'criterion':t.suggest_categorical('criterion',['gini','entropy'])}
    fs=scores(p); t.set_user_attr('fold_scores',fs); return float(np.mean(fs))
def save_trial(study, trial):
    rows=[]
    for tr in study.trials: rows.append({'trial':tr.number,'state':tr.state.name,'cv_macro_f1':tr.value,'fold_scores':json.dumps(tr.user_attrs.get('fold_scores',[])),**tr.params})
    pd.DataFrame(rows).to_csv(OUT/'optimization_trials.csv',index=False)
if resume_validated and all((OUT/f).exists() for f in ['optimization_trials.csv','best_parameters.json','best_cv_score.json']):
    td=pd.read_csv(OUT/'optimization_trials.csv'); bestp=json.loads((OUT/'best_parameters.json').read_text()); best_data=json.loads((OUT/'best_cv_score.json').read_text()); best_cv=float(best_data['mean_trip_grouped_cv_macro_f1']); best_trial_number=int(best_data['trial'])
else:
    study=optuna.create_study(direction='maximize',sampler=TPESampler(seed=SEED,n_startup_trials=1)); study.optimize(objective,n_trials=N_TRIALS,catch=(Exception,),callbacks=[save_trial])
    trials=[{'trial':t.number,'state':t.state.name,'cv_macro_f1':t.value,'fold_scores':json.dumps(t.user_attrs.get('fold_scores',[])),**t.params} for t in study.trials]
    td=pd.DataFrame(trials); td.to_csv(OUT/'optimization_trials.csv',index=False)
    best=study.best_trial; bestp={**best.params,'class_weight':'balanced','n_jobs':-1,'random_state':SEED}; best_cv=float(best.value); best_trial_number=best.number
    (OUT/'best_parameters.json').write_text(json.dumps(bestp,indent=2)); (OUT/'best_cv_score.json').write_text(json.dumps({'mean_trip_grouped_cv_macro_f1':best.value,'fold_scores':best.user_attrs['fold_scores'],'trial':best.number},indent=2))
complete=td[td.state.eq('COMPLETE')]; failed=td[td.state.eq('FAIL')]
if complete.empty: raise RuntimeError('No successful optimization trials')
print('completed',len(complete),'failed',len(failed),'best',bestp,best_cv)


completed 3 failed 0 best {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 5, 'max_features': 'log2', 'criterion': 'gini', 'class_weight': 'balanced', 'n_jobs': -1, 'random_state': 42} 0.8425999655720612


## Final fit and student-testing comparison

Expected student inputs are processed trip feature files and monotonic-alignment CSVs under `data/processed/student_test/`, generated using the same established candidate and feature pipeline. The evaluator partition is not used. The setup prepares missing student feature/alignment tables using the existing project pipeline, then the notebook evaluates the held-out student partition only after tuning.

In [3]:
if RESUME_PILOT_CHECKPOINTS and (OUT/'untuned_vs_tuned_comparison.csv').exists() and (OUT/'optimization_summary.json').exists() and (ROOT/'reports'/'section_4_bayesian_optimization.md').exists():
    comp=pd.read_csv(OUT/'untuned_vs_tuned_comparison.csv')
    print('Resumed saved pilot evaluation; model fits and optimization trials were not rerun.')
    display(comp)
else:
    student=load_labelled(STUDENT,'student-testing'); SX=features(student); sy=student.label.astype(int)
    # Candidate recall uses the complete student reference observation set; shared candidate coverage is identical for both models.
    allstu=[]
    for p in sorted(STUDENT.glob('trip_*_features.csv')):
        tid=p.stem.replace('trip_','').replace('_features',''); ap=STUDENT/f'trip_{tid}_monotonic_alignment.csv'
        if ap.exists():
            z=pd.read_csv(p).merge(pd.read_csv(ap)[['trip_id','gps_index','aligned_arc_id']],on=['trip_id','gps_index'],how='inner',validate='many_to_one'); z['label']=(z.arc_id==z.aligned_arc_id).astype(int); allstu.append(z)
    allstu=pd.concat(allstu,ignore_index=True); candidate_recall=float(allstu.groupby(['trip_id','gps_index']).label.max().mean())
    def evaluate(name,p):
        m=model(p).fit(X,y); start=time.perf_counter(); pred=m.predict(SX); prob=m.predict_proba(SX)[:,1]; infer=time.perf_counter()-start
        rank=student[['trip_id','gps_index','arc_id','label','distance_m']].copy(); rank['prob']=prob; rank=rank.sort_values(['prob','distance_m'],ascending=[False,True]).drop_duplicates(['trip_id','gps_index'])
        return {'model':name,'cv_macro_f1_mean':base_cv if name.startswith('Untuned') else best_cv,'student_top1_accuracy':accuracy_score(rank.label,np.ones(len(rank),dtype=int)),'student_macro_f1':f1_score(sy,pred,average='macro',zero_division=0),'student_log_loss':log_loss(sy,prob,labels=[0,1]),'inference_time_seconds':infer,'candidate_recall':candidate_recall}
    untuned=evaluate('Untuned Random Forest',baseline); tuned=evaluate('Tuned Random Forest',bestp); comp=pd.DataFrame([untuned,tuned])
    pd.DataFrame([untuned]).to_csv(OUT/'baseline_results.csv',index=False); pd.DataFrame([tuned]).to_csv(OUT/'tuned_model_results.csv',index=False); comp.to_csv(OUT/'untuned_vs_tuned_comparison.csv',index=False)
    ax=comp.set_index('model')[['cv_macro_f1_mean','student_top1_accuracy','student_macro_f1','student_log_loss']].plot(kind='bar',figsize=(10,5)); plt.tight_layout(); plt.savefig(OUT/'untuned_vs_tuned_metric_comparison.png',dpi=160); plt.show()
    ax=comp.set_index('model')[['inference_time_seconds']].plot(kind='bar',figsize=(6,4)); plt.tight_layout(); plt.savefig(OUT/'inference_time_comparison.png',dpi=160); plt.show()
    summary={'objective':'maximize mean trip-grouped CV Macro F1','sampler':'Optuna TPE','planned_trials':30,'run_trial_limit':N_TRIALS,'experiment_scope':'time-bounded pilot subset; six size-stratified training trips','pilot_training_trip_ids':PILOT_IDS,'completed_trials':len(complete),'failed_trials':len(failed),'best_trial':best_trial_number,'best_cv_macro_f1':best_cv,'baseline_cv_macro_f1':base_cv,'folds':fold_ids,'student_testing_used_for_tuning':False,'evaluator_testing_accessed':False,'candidate_recall':candidate_recall}
    (OUT/'optimization_summary.json').write_text(json.dumps(summary,indent=2)); (ROOT/'reports').mkdir(exist_ok=True)
    report=f"""# Section 4: Bayesian Hyperparameter Optimisation — Resource-Limited Pilot

    1. **Hyperparameters:** n_estimators, max_depth, min_samples_split, min_samples_leaf, max_features, criterion.
    2. **Pilot search distributions:** n_estimators integer-uniform 50–150 step 50; max_depth categorical {None, 4, 8, 12, 16}; min_samples_split integer-uniform 2–10; min_samples_leaf integer-uniform 1–5; max_features categorical {sqrt, log2, 0.5}; criterion categorical {gini, entropy}.
    3. **Objective:** maximise mean Macro F1 over three trip-grouped folds on six pilot training trips.
    4. **Trials:** {len(complete)} completed, {len(failed)} failed, 0 pruned (30 planned; capped at 3 pilot trials after full-data CV exceeded the 30-minute budget).
    5. **Best values:** `{json.dumps(bestp)}`.
    6. **Improvement:** CV objective change {tuned['cv_macro_f1_mean']-untuned['cv_macro_f1_mean']:+.6f}.

    ## Tuned versus untuned

    | Model | CV Macro F1 | Student Top-1 | Student Macro F1 | Student Log Loss | Inference (s) | Candidate Recall |
    |---|---:|---:|---:|---:|---:|---:|
    | {untuned['model']} | {untuned['cv_macro_f1_mean']:.6f} | {untuned['student_top1_accuracy']:.6f} | {untuned['student_macro_f1']:.6f} | {untuned['student_log_loss']:.6f} | {untuned['inference_time_seconds']:.6f} | {candidate_recall:.6f} |
    | {tuned['model']} | {tuned['cv_macro_f1_mean']:.6f} | {tuned['student_top1_accuracy']:.6f} | {tuned['student_macro_f1']:.6f} | {tuned['student_log_loss']:.6f} | {tuned['inference_time_seconds']:.6f} | {candidate_recall:.6f} |

    This is a six-trip pilot, not a full-data result. The 80-trip grouped baseline and final evaluation remain incomplete. The untuned model retains the official 300-tree configuration. Student-test metrics were computed after tuning on identical candidate rows and reference alignments. Candidate recall is a shared property of those candidates. The imputer is fit per fold; a preceding global unsupervised feature-selection analysis was not nested, which may introduce mild transductive feature-selection information if it informed the hard-coded feature list. Fold assignments are saved in `outputs/hyperparameter_optimization/optimization_summary.json`.

    Reproduce by installing project requirements (Optuna 5.1.0), setting `RAW` to the directory containing only the permitted `train` and `student_test` partitions, then running all notebook cells from the repository root. The existing pipeline prepares missing student features/alignments.
    """
    (ROOT/'reports'/'section_4_bayesian_optimization.md').write_text(report,encoding='utf8'); print(comp.to_string(index=False))


Resumed saved pilot evaluation; model fits and optimization trials were not rerun.


,model,cv_macro_f1_mean,student_top1_accuracy,student_macro_f1,student_log_loss,inference_time_seconds,candidate_recall
0,Untuned Random Forest,0.829546,0.812063,0.878550,0.068577,9.371431,0.999478
1,Tuned Random Forest,0.842600,0.810496,0.871397,0.083751,3.191610,0.999478
